# Glaciers selection based on RGI6 

In [ ]:
# basic
from tqdm import tqdm
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import os

# spatial
import geopandas as gpd
import xarray as xr
import rioxarray as rio
import xrspatial as xrs
from exactextract import exact_extract

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes')
os.chdir(wd)

# import config 
with open(wd / "config.yml") as f:
    cfg = yaml.safe_load(f)

# aggregation
from src.utils import extract_timeseries, extract_attributes

In [ ]:
# basins
basins = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg")

## Dynamic inputs

In [ ]:
# Meteorological forcings (ERA5)
variables = ["prcp", "tasmax", "tasmin", "tas", "ep"]

for variable in tqdm(variables):

    # Load only the yearly subset using time selection
    file_path = os.path.join(cfg["dir"]["data"], f"ERA5/{variable}_ERA5_1960_2024.nc")

    with xr.open_dataset(file_path) as climate:
        climate = climate[variable]
        data = extract_timeseries(climate, basins, time_period=("1990-01-01", cfg["time"]["ref_end"]))
        data.to_parquet(f"data/climate/HydroAtlas_{variable}_ERA5_1990_2024.parquet", index=True)

## Static inputs
Same attributes as in AndeanGC

In [ ]:
# basic attributes
basins_projected = basins.to_crs(32719)  # UTM zone 19S for 
basins["basin_area"] = basins_projected.area / 1e6  # from m2 to km2
basins["basin_cenlat"] = basins_projected.centroid.to_crs(4326).y
basins["basin_cenlon"] = basins_projected.centroid.to_crs(4326).x
basins["total_perim"] = basins_projected.geometry.length / 1e3  # from m to km
basins["gc_index"] = basins.total_perim/(2*(np.sqrt(np.pi*basins.basin_area)))

In [ ]:
# elevation features (NASADEM)
dem = xr.open_dataset(cfg["dir"]["data"] + "/DEM/DEM_Andes_270m.tif").band_data.sel(band=1, drop = True)
dem = dem.sel(x=slice(basins.total_bounds[0], basins.total_bounds[2]), 
              y=slice(basins.total_bounds[3], basins.total_bounds[1]))
dem  = dem.fillna(0)  # NAs to sea level (= 0)

slope  = xrs.slope(dem.rio.reproject("EPSG:32719"))
slope  = slope.rio.reproject("EPSG:4326")
aspect = xrs.aspect(dem)

basins = extract_attributes(dem, basins, "elev_mean", fun = "mean")
basins = extract_attributes(dem, basins, "elev_median", fun = "median")
basins = extract_attributes(dem, basins, "elev_std", fun = "stdev")
basins = extract_attributes(slope, basins, "slope_mean", fun = "mean")
basins = extract_attributes(aspect, basins, "aspect_mean", fun = "mean")

In [ ]:
# climate attributes (ERA5)
ref_period = ["1990-01-01", "2019-12-31"]

climate_stack = xr.open_mfdataset(cfg["dir"]["data"] + "/ERA5/*_ERA5_1960_2024.nc")
climate_stack = climate_stack.sel(time=slice(ref_period[0], ref_period[1]))
climate_stack['tas'] = (climate_stack.tasmax + climate_stack.tasmin) / 2

# prcp_mean
stack = climate_stack.prcp.resample(time='YS').sum(dim='time').mean(dim='time')
basins = extract_attributes(stack, basins, "prcp_mean_ERA5",  fun = "mean")

# prcp_pci
stack = climate_stack.prcp.resample(time='MS').sum(dim='time').groupby('time.month').mean(dim='time')
stack = (stack ** 2).sum(dim='month') * 100 / (stack.sum(dim='month') ** 2)
basins = extract_attributes(stack, basins, "prcp_pci_ERA5", fun = "mean")

# ev_mean and aridity
stack = climate_stack.ep.resample(time='YS').sum(dim='time').mean(dim='time')
basins = extract_attributes(stack, basins, "ev_mean_ERA5", fun = "mean")
basins['aridity_ERA5'] = basins['prcp_mean_ERA5'] / basins['ev_mean_ERA5']

# solid prcp (< 0◦C)
stack = xr.where(climate_stack.tas > 0, 0, climate_stack.prcp)
stack = stack.resample(time='YS').sum(dim='time').mean(dim='time')
basins = extract_attributes(stack, basins, "solid_prcp_ERA5", fun = "mean")
basins['frac_snow_ERA5'] = basins['solid_prcp_ERA5'] / basins['prcp_mean_ERA5']

# prcp freq
stack = (climate_stack.prcp < 1).sum(dim='time')
stack = climate_stack.prcp.sizes['time'] / stack
basins = extract_attributes(stack, basins, "low_prec_freq_ERA5", fun = "mean")

stack = (climate_stack.prcp > climate_stack.prcp.mean(dim='time') * 5).sum(dim='time')
stack = climate_stack.prcp.sizes['time'] / stack
basins = extract_attributes(stack, basins, "high_prec_freq_ERA5", fun = "mean")

In [ ]:
# glacier attributes

# dhdt in m/yr
dhdt = rio.open_rasterio(cfg["dir"]["data"] + "/GLACIERS/dhdt_2000_2020_hugonnet.tif").sel(band=1, drop = True) # in m/yr
basins = extract_attributes(dhdt, basins, "glacier_dhdt", fun = "mean")

# vol in km3 (M22)
vol = rio.open_rasterio(cfg["dir"]["data"] + "/GLACIERS/volume_millan_2022_100m.tif").sel(band=1, drop = True) # in km3
basins = extract_attributes(vol, basins, "glacier_volume_M22", fun = "sum")

# volume in km3 (F19)
vol = rio.open_rasterio(cfg["dir"]["data"] + "/GLACIERS/volume_Farinotti_2019_200m.tif").sel(band=1, drop = True) # in km3
basins = extract_attributes(vol, basins, "glacier_volume_F19", fun = "sum")

In [ ]:
# water demand features
ref_period = ["1990-01-01", "2019-12-31"]

# dams 
dams_data = gpd.read_file(cfg["dir"]["data"] + "/LAND_COVER/DAMS/GDAT_V1_DAMS.gpkg") 
dams_data = dams_data[dams_data.Capac > 100] # only dams with capacity larger than 100 hm3

dams_count = gpd.sjoin(dams_data, basins.reset_index(), how='inner', predicate='within')
dams_count = dams_count.groupby('gauge_id').size()
basins = basins.join(dams_count.rename("dams_count"))
basins["dams_count"] = basins["dams_count"].fillna(0)
    
# water use (Huang et al. 2017)
wu_global   = xr.open_dataset(cfg["dir"]["data"] + "/WATER_USE/Huang_HESS/Water_use_Huang_HESS.nc")
wu_global   = wu_global.sel(date=slice(ref_period[0], ref_period[1]))
wu_global   = wu_global.resample(date ="YS").sum().mean(dim="date")

basins = extract_attributes(wu_global.cons_irr, basins, "cons_irrigation",  fun = "mean")
basins = extract_attributes(wu_global.cons_total, basins, "cons_total",  fun = "mean")

In [ ]:
# land cover features

land_cover_data = rio.open_rasterio(cfg["dir"]["data"] + "/LAND_COVER/GLOBAL/PROBAV_LC100_global_v3.0.1_2019.tif").sel(band=1, drop = True)
land_cover_data = land_cover_data.sel(x=slice(basins.total_bounds[0], basins.total_bounds[2]), 
                                      y=slice(basins.total_bounds[3], basins.total_bounds[1]))

land_cover_data = land_cover_data.where(~((land_cover_data >= 100) & (land_cover_data <= 200)), 10)
land_cover_data = exact_extract(land_cover_data, basins, ["unique", "frac"], progress=False, 
                                output = "pandas", include_cols=["gauge_id"])

land_cover_data = land_cover_data.apply(lambda row: pd.Series(dict(zip(row['unique'], row['frac']))), axis=1)
land_cover_data = land_cover_data.set_index(basins.gauge_id).fillna(0) * 100

basins = basins.set_index("gauge_id")
basins["forest_cover"] = land_cover_data[10]
basins["shrubs_veg_cover"] = land_cover_data[20]
basins["herbaceous_veg_cover"] = land_cover_data[30]
basins["cropland_cover"] = land_cover_data[40]
basins["urban_cover"] = land_cover_data[50]
basins["sparse_veg_cover"] = land_cover_data[60]
basins["water_cover"] = land_cover_data[80]

In [ ]:
# Save to file
basins.to_file('data/hydro/basins_hydroATLAS_lev08.gpkg')